In [4]:
import torch
import matplotlib.pyplot as plt

In [5]:
torch.manual_seed(42)

print(torch.__version__)

2.8.0+cu128


In [6]:
x = torch.tensor([[0.0],
                  [0.0],
                  [0.0],
                  [7.0],
                  [0.0],
                  [0.0],
                  [0.0]])
y = torch.tensor([[7.0]])
    
print(x.shape)
print(y.shape)

torch.Size([7, 1])
torch.Size([1, 1])


In [7]:
hidden_size = 1
Wx = torch.randn(1, 1, requires_grad=True)
Wh = torch.randn(1, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
Wy = torch.randn(1, 1, requires_grad=True)
by = torch.zeros(1, requires_grad=True)

In [8]:
def forward(x):
    h = torch.zeros(1)
    states = []

    for t in range(len(x)):
        h = torch.tanh(Wx @ x[t] + Wh @ h + b)
        states.append(h)

    y_hat = Wy @ h + by

    return y_hat, states

In [10]:
y_hat, states = forward(x)

print('Predizione:', y_hat.item())
print('Stati:')

for t, h in enumerate(states):
    print(t, h.item())

Predizione: 0.0004895258462056518
Stati:
0 0.0
1 0.0
2 0.0
3 0.9822158813476562
4 0.12584789097309113
5 0.01620897278189659
6 0.0020878652576357126


In [11]:
def loss_fn(y_hat, y):
    return((y_hat - y) ** 2).mean()

In [12]:
lr = 0.05

for epoch in range(1000):
    y_hat, states = forward(x)
    loss = loss_fn(y_hat, y)
    loss.backward()

    with torch.no_grad():
        Wx -= lr * Wx.grad
        Wh -= lr * Wh.grad
        b -= lr * b.grad
        Wy -= lr * Wy.grad
        by -= lr * by.grad

    Wx.grad.zero_()
    Wh.grad.zero_()
    b.grad.zero_()
    Wy.grad.zero_()
    by.grad.zero_()

    if epoch % 100 == 0:
        print(epoch, loss.item())

0 48.99314498901367
100 9.094947017729282e-13
200 9.094947017729282e-13
300 9.094947017729282e-13
400 9.094947017729282e-13
500 9.094947017729282e-13
600 9.094947017729282e-13
700 9.094947017729282e-13
800 9.094947017729282e-13
900 9.094947017729282e-13


In [13]:
y_hat, states = forward(x)

print('Predizione finale:', y_hat.item())

for t, h in enumerate(states):
    print(f"h[{t}] = ", h.item())

Predizione finale: 6.999999046325684
h[0] =  0.8570494651794434
h[1] =  0.9654352068901062
h[2] =  0.9712374806404114
h[3] =  0.9997419714927673
h[4] =  0.972865641117096
h[5] =  0.9715980887413025
h[6] =  0.9715368747711182


In [14]:
def forward_with_perturbation(x, perturb_t=None, epsilon=0.0):
    h = torch.zeros(1)

    states = []

    for t in range(len(x)):
        h = torch.tanh(Wx @ x[t] + Wh @ h + b)

        if t == perturb_t:
            h = h + epsilon

        states.append(h)

    y_hat = Wy @ h + by

    return y_hat, states

In [15]:
normal_prediction, _ = forward_with_perturbation(x)

perturbed_prediction, _ = forward_with_perturbation(x, perturb_t=3, epsilon=0.01)

print("Normale:", normal_prediction.item())
print("Perturbata:", perturbed_prediction.item())
print("Differenza:", perturbed_prediction.item() - normal_prediction.item())

Normale: 6.999999046325684
Perturbata: 7.000001907348633
Differenza: 2.86102294921875e-06


In [16]:
epsilon = 0.01

for t in range(len(x)):
    normal, _ = forward_with_perturbation(x)
    perturbed, _ = forward_with_perturbation(x, perturb_t=t, epsilon=epsilon)
    delta = perturbed.item() - normal.item()

    print(f"h[{t}] -> delta prediction = {delta:.6f}")

h[0] -> delta prediction = 0.000000
h[1] -> delta prediction = 0.000000
h[2] -> delta prediction = 0.000000
h[3] -> delta prediction = 0.000003
h[4] -> delta prediction = 0.000062
h[5] -> delta prediction = 0.001297
h[6] -> delta prediction = 0.027073


In [17]:
y_hat, states = forward_with_perturbation(x)
loss = loss_fn(y_hat, y)

gradients = torch.autograd.grad(y_hat, states, retain_graph=True)

for t, grad in enumerate(gradients):
    print(f"h[{t}] -> dy/dh = {grad.item():.8f}")

h[0] -> dy/dh = 0.00000000
h[1] -> dy/dh = 0.00000001
h[2] -> dy/dh = 0.00000013
h[3] -> dy/dh = 0.00029101
h[4] -> dy/dh = 0.00631132
h[5] -> dy/dh = 0.13085490
h[6] -> dy/dh = 2.70730877


In [21]:
epsilon = 0.01

for t in range(len(x)):
    normal, _ = forward_with_perturbation(x)
    perturbed, _ = forward_with_perturbation(x, perturb_t=t, epsilon=epsilon)
    numerical = (perturbed.item() - normal.item()) / epsilon
    analytical = gradients[t].item()

    print(f"h[{t}] | "
    f"numerical = {numerical:.8f} | "
    f"gradient = {analytical:.8f}")

h[0] | numerical = 0.00000000 | gradient = 0.00000000
h[1] | numerical = 0.00000000 | gradient = 0.00000001
h[2] | numerical = 0.00000000 | gradient = 0.00000013
h[3] | numerical = 0.00028610 | gradient = 0.00029101
h[4] | numerical = 0.00624657 | gradient = 0.00631132
h[5] | numerical = 0.12974739 | gradient = 0.13085490
h[6] | numerical = 2.70729065 | gradient = 2.70730877
